In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
# Residual blocks prevent vanishing gradients, reduces overfitting and allows for faster training.
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(
            out_channels
        )  # Batch normalisation makes training more stable, allows for higher learning rates and acts as regularistion.

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # If channel counts differ, project the skip
        self.projection = None
        if in_channels != out_channels:
            self.projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # match channels if needed
        if self.projection is not None:
            identity = self.projection(identity)

        out += identity
        return F.relu(out)

In [ ]:
# The Contracting Path (encoding)
class Encoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.res_block = ResidualBlock(in_channels, out_channels)
        self.max_pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        skip = self.res_block(x)
        down = self.max_pool(skip)
        return skip, down

In [ ]:
# The Expanding Path (decoding)
class UpBlock(nn.Module):
    def __init__(self, decoder_channels, skip_channels, out_channels):
        super().__init__()
        self.res_block = ResidualBlock(out_channels + skip_channels, out_channels)
        self.up_convolution = nn.ConvTranspose2d(
            decoder_channels, out_channels, kernel_size=2, stride=2
        )

    def forward(self, x, skip):
        x = self.up_convolution(x)
        if x.size() != skip.size():
            x = F.interpolate(
                x, size=skip.shape[2:], mode="bilinear", align_corners=True
            )
        x = torch.cat([x, skip], dim=1)
        return self.res_block(x)

In [17]:
class UNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # down blocks
        self.down1 = Encoder(1, 64)
        self.down2 = Encoder(64, 128)
        self.down3 = Encoder(128, 256)
        self.down4 = Encoder(256, 512)

        # bottleneck
        self.bottleneck = ResidualBlock(512, 1024)

        # up blocks
        self.up1 = UpBlock(1024, 512, 512)
        self.up2 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 128)
        self.up4 = UpBlock(128, 64, 64)

        # final layer
        self.final_conv = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        # encoder
        skip1, x = self.down1(x)
        skip2, x = self.down2(x)
        skip3, x = self.down3(x)
        skip4, x = self.down4(x)

        # bottleneck
        x = self.bottleneck(x)

        # decoder
        x = self.up1(x, skip4)
        x = self.up2(x, skip3)
        x = self.up3(x, skip2)
        x = self.up4(x, skip1)

        # output
        return self.final_conv(x)

In [ ]:
def dice_loss(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)  # bound predictions to [0, 1]

    pred = pred.contiguous()
    target = target.contiguous()

    intersection = (pred * target).sum(dim=(2, 3))
    denom = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))

    dice = (2.0 * intersection + smooth) / (denom + smooth)
    return 1 - dice.mean()


class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        dice = dice_loss(pred, target)
        return bce_loss + dice